# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [12]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
gemma_api_key=os.getenv('GEMMA_API_KEY')
nvidia_api_key=os.getenv('NVIDIA_API_KEY')


In [3]:
# Connect to client libraries



anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

openai = OpenAI(api_key=openai_api_key, base_url=openrouter_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
nvidia = OpenAI(api_key=nvidia_api_key, base_url=openrouter_url)
gemma = OpenAI(api_key=gemma_api_key,base_url=openrouter_url)

In [23]:
models = ["gpt-oss-20b:free", "qwen2.5-coder", "nvidia/nemotron-3-ultra-550b-a55b:free", "gemini-3-flash-lite-preview","gemma-4-31b-it:free" ]

clients = {"gpt-oss-20b:free": openai,"gemini-3-flash-lite-preview": gemini, "qwen2.5-coder": ollama, "nvidia/nemotron-3-ultra-550b-a55b:free": nvidia,"gemma-4-31b-it:free":gemma}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [10]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.19045',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget', 'choco'],
 'cpu': {'brand': 'AMD Ryzen 5 5600H with Radeon Graphics',
  'cores_logical': 12,
  'cores_physical': 6,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [13]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model="gpt-oss-20b:free", messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

**Short answer**

Yes – you don’t have a C++ compiler installed yet.  
After installing one, you can compile and run `main.cpp` with a tiny Python snippet like this:

```python
import subprocess

# 1. Compile
compile_command = ["g++", "-O3", "-std=c++20", "main.cpp", "-o", "main.exe"]
subprocess.run(compile_command, check=True, text=True, capture_output=True)

# 2. Run
run_command = ["./main.exe"]          # or ["main.exe"] on Windows
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)               # the program’s output
```

If you prefer MSVC (`cl.exe`) instead of GCC, replace the two lists with:

```python
compile_command = ["cl", "/EHsc", "/O2", "main.cpp", "/Fe:main.exe"]
run_command     = ["main.exe"]
```

---

## 1. Install a C++ compiler

Your system reports that **no compiler is present** (`gcc`, `g++`, `clang`, `msvc_cl` are all empty).  
The easiest way on Windows 10 is to use **winget** (or Chocolatey) to install a pre‑built toolchain.

### Option A – MinGW‑w64 (GCC)

```powershell
# Open PowerShell as Administrator
winget install -e --id=mingw-w64
```

* `mingw-w64` installs `g++`, `gcc`, `make`, etc.  
* The installer automatically adds the `bin` directory to your `PATH`, so you can run `g++ --version` immediately.

### Option B – MSVC Build Tools (cl.exe)

```powershell
# Open PowerShell as Administrator
winget install -e --id=Microsoft.VisualStudio.2022.BuildTools
```

* This installs the Visual Studio 2022 Build Tools, including `cl.exe`.  
* After installation you’ll need to run your scripts from the **Developer Command Prompt for VS 2022** (or add the `VC\Tools\MSVC\<version>\bin\Hostx64\x64` folder to `PATH` manually).

> **Tip** – If you only need a single‑file compiler, MinGW‑w64 is the quickest path.  
> If you already have Visual Studio installed, you can skip the installation step entirely.

---

## 2. Verify the installation

Open a **new** terminal (PowerShell, CMD, or the VS Developer Prompt) and type:

```powershell
# For GCC
g++ --version

# For MSVC
cl
```

You should see a version string. If not, double‑check that the `PATH` was updated (restart the terminal).

---

## 3. Compile & run from Python

Below are the exact command lists you can drop into your Python script.  
Both examples assume `main.cpp` lives in the current working directory.

| Compiler | Compile command (list) | Run command (list) |
|----------|------------------------|--------------------|
| **GCC**  | `["g++", "-O3", "-std=c++20", "main.cpp", "-o", "main.exe"]` | `["./main.exe"]` (or `["main.exe"]` on Windows) |
| **MSVC** | `["cl", "/EHsc", "/O2", "main.cpp", "/Fe:main.exe"]` | `["main.exe"]` |

### Full Python example

```python
import subprocess

# --- Compile ----------------------------------------------------
compile_command = ["g++", "-O3", "-std=c++20", "main.cpp", "-o", "main.exe"]
subprocess.run(compile_command, check=True, text=True, capture_output=True)

# --- Execute ----------------------------------------------------
run_command = ["./main.exe"]          # or ["main.exe"] on Windows
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

print(run_result.stdout)               # prints the program’s output
```

> **Why these flags?**  
> * `-O3` / `/O2` – aggressive optimisation for the fastest runtime.  
> * `-std=c++20` – enables the latest C++ standard (adjust if you need an older one).  
> * `/EHsc` – enables standard C++ exception handling in MSVC.

---

## 4. Quick sanity test

Create a minimal `main.cpp`:

```cpp
#include <iostream>

int main() {
    std::cout << "Hello, world!\n";
    return 0;
}
```

Run the Python snippet above – you should see `Hello, world!` printed.

---

### Recap

1. **Install a compiler** (MinGW‑w64 or MSVC Build Tools).  
2. **Verify** with `g++ --version` or `cl`.  
3. Use the provided `compile_command` / `run_command` lists in your Python script.  

That’s all you need to compile and execute a single C++ file on your Windows 10 machine. Happy coding!

In [14]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]


## And now, on with the main task

In [15]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [16]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [17]:
def write_output(cpp):
    with open("main.cpp", "w") as f:
        f.write(cpp)

In [18]:
def port(model, python):
    client = clients[model]
    openai_reasoning_models = {"gpt-oss-20b:free"}
    reasoning_effort = "high" if model in openai_reasoning_models else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)
    return reply

In [19]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [20]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [21]:
def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "e:\python 314\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "e:\python 314\Lib\site-packages\gradio\route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "e:\python 314\Lib\site-packages\gradio\blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "e:\python 314\Lib\site-packages\gradio\blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
   

In [ ]:
compile_and_run()

Qwen 2.5 Coder: Fail  
DeepSeek Coder v2: 0.114050084  
OpenAI gpt-oss 20B: 0.080438  
Qwen 30B: 0.113734  
OpenAI gpt-oss 120B: 1.407383




In Ed's experiments, the performance speedups were:

9th place: Qwen 2.5 Coder: Fail  
8th place: OpenAI GPT-OSS 120B: 14X speedup    
7th place: DeepSeek Coder v2: 168X speedup  
6th place: Qwen3 Coder 30B: 168X speedup   
5th place: Claude Sonnet 4.5: 184X speedup   
4th place: GPT-5: 233X speedup  
**3rd place: oss-20B: 238X speedup**  
2nd place: Grok 4: 1060X speedup  
1st place: Gemini 2.5 Pro: 1440X speedup  